In [12]:
import copy
import os
import tempfile
import numpy as np
import pandas as pd
import wntr
from wntr.epanet.util import to_si, FlowUnits, HydParam
import json
import time
import hashlib
from collections import Counter
import networkx as nx

In [13]:
path = r"C:\Data\pipeline-leakage-system\Phase 1\EPANET\ky9.inp"
wn = wntr.network.WaterNetworkModel(path)

EMITTER_C_TO_SI = to_si(FlowUnits.GPM, 1.0, HydParam.EmitterCoeff)
GPM_TO_M3S = 0.0000630902
FT_TO_M = 0.3048

In [14]:
base_wn = copy.deepcopy(wn)

### Getting detils of the pipeline network

In [15]:
test_wn = wntr.network.WaterNetworkModel(path)

print("Nodes:", test_wn.node_name_list)
print("Links:", test_wn.link_name_list)
print("Junctions:", test_wn.junction_name_list)
print("Reservoirs:", test_wn.reservoir_name_list)
print("Pipes:", test_wn.pipe_name_list)
print("Valves:", test_wn.valve_name_list)
print("Tanks:", test_wn.tank_name_list)
print("Pumps:", test_wn.pump_name_list)

print("\nPipe details:")
for name in test_wn.pipe_name_list:
    p = test_wn.get_link(name)
    print(name, ":", p.start_node_name, "->", p.end_node_name,
          "| L =", p.length, "| D =", p.diameter, "| C =", p.roughness)

Nodes: ['J-1', 'J-10', 'J-100', 'J-1001', 'J-1002', 'J-1003', 'J-1004', 'J-1005', 'J-1006', 'J-1007', 'J-1008', 'J-101', 'J-1010', 'J-1011', 'J-1012', 'J-1013', 'J-1014', 'J-1015', 'J-1016', 'J-1017', 'J-1018', 'J-1019', 'J-102', 'J-1020', 'J-1021', 'J-1022', 'J-1023', 'J-1024', 'J-1025', 'J-1026', 'J-1027', 'J-1028', 'J-1029', 'J-103', 'J-1030', 'J-1031', 'J-1032', 'J-1033', 'J-1034', 'J-1035', 'J-1036', 'J-1037', 'J-1038', 'J-1039', 'J-104', 'J-1041', 'J-1042', 'J-1044', 'J-1045', 'J-1046', 'J-1047', 'J-1048', 'J-1049', 'J-105', 'J-1050', 'J-1051', 'J-1052', 'J-1053', 'J-1054', 'J-1055', 'J-1056', 'J-1057', 'J-1058', 'J-1059', 'J-106', 'J-1060', 'J-1061', 'J-1062', 'J-1064', 'J-1065', 'J-1066', 'J-1067', 'J-1068', 'J-1069', 'J-107', 'J-1070', 'J-1071', 'J-1072', 'J-1073', 'J-1074', 'J-1075', 'J-1077', 'J-1078', 'J-1079', 'J-108', 'J-1081', 'J-1082', 'J-1083', 'J-1084', 'J-1085', 'J-1086', 'J-1087', 'J-1089', 'J-109', 'J-1090', 'J-1091', 'J-1092', 'J-1093', 'J-1094', 'J-1095', 'J-1096

### Inspecting EPANET and WNTR unit/convention

In [16]:
# ---------------------------------------------------------
# 1. EPANET conventions used by THIS network
# ---------------------------------------------------------
hyd = wn.options.hydraulic

print("=" * 70)
print("EPANET CONVENTIONS — CURRENT NETWORK")
print("=" * 70)

print("INP flow units       :", hyd.inpfile_units)
print("INP pressure units   :", hyd.inpfile_pressure_units)
print("Headloss formula     :", hyd.headloss)
print("Demand model         :", hyd.demand_model)
print("Emitter exponent     :", hyd.emitter_exponent)
print("Specific gravity     :", hyd.specific_gravity)

# ---------------------------------------------------------
# 2. All EPANET-supported flow units
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("EPANET FLOW UNITS")
print("=" * 70)

for unit in FlowUnits:
    print(
        f"{unit.name:6} | "
        f"traditional={unit.is_traditional} | "
        f"metric={unit.is_metric} | "
        f"SI factor={unit.factor}"
    )

# ---------------------------------------------------------
# 3. EPANET parameter categories handled by WNTR
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("EPANET PARAMETERS CONVERTED BY WNTR")
print("=" * 70)

for param in HydParam:
    print(f"{param.name:20} | {param}")

EPANET CONVENTIONS — CURRENT NETWORK
INP flow units       : GPM
INP pressure units   : None
Headloss formula     : H-W
Demand model         : DDA
Emitter exponent     : 0.5
Specific gravity     : 1.0

EPANET FLOW UNITS
CFS    | traditional=True | metric=False | SI factor=0.0283168466
GPM    | traditional=True | metric=False | SI factor=6.30901964e-05
MGD    | traditional=True | metric=False | SI factor=0.043812636388888895
IMGD   | traditional=True | metric=False | SI factor=0.05261678240740741
AFD    | traditional=True | metric=False | SI factor=0.014276410185185185
LPS    | traditional=False | metric=True | SI factor=0.001
LPM    | traditional=False | metric=True | SI factor=1.6666666666666667e-05
MLD    | traditional=False | metric=True | SI factor=0.011574074074074073
CMH    | traditional=False | metric=True | SI factor=0.0002777777777777778
CMD    | traditional=False | metric=True | SI factor=1.1574074074074073e-05
SI     | traditional=False | metric=False | SI factor=1.0

EPANET 

# Run Simulation

In [17]:
def run_simulation(model, hours=24):
    """
    Run EPANET hydraulics using only the current in-memory WNTR model.

    No permanent .inp file is written.
    EPANET's temporary files are created inside a temporary directory
    and removed automatically after the simulation.
    """

    model.options.time.duration = int(hours * 3600)
    model.options.time.hydraulic_timestep = 3600
    model.options.time.pattern_timestep = 3600
    model.options.time.report_timestep = 3600

    with tempfile.TemporaryDirectory() as tmpdir:

        results = wntr.sim.EpanetSimulator(model).run_sim(
            file_prefix=os.path.join(tmpdir, "scenario"),
            convergence_error=True
        )

    return results

# Generate Scenarios + make Dataframe

### 1. configuration, expected dimensions, safety guards

In [18]:
# ---------------------------------------------------------------------
# Reproducibility / size of the run
# ---------------------------------------------------------------------
SEED = 42
N_SCENARIOS = 500             # number of scenarios to generate
SIM_HOURS = 24                # 24 h simulation, 1 h hydraulic/report step
MAX_ATTEMPTS_PER_SCENARIO = 12  # resample a scenario if EPANET fails / gate rejects it

# ---------------------------------------------------------------------
# Expected dimensions of the network loaded in this notebook (ky9.inp)
# ---------------------------------------------------------------------
EXPECTED = dict(junctions=1242, pipes=1270, reservoirs=4, tanks=15, pumps=17, valves=56)
EXPECTED_INPUT_DIM = 3755     # 1242 (demand) + 1242 (pressure) + 1270 (flow) + 1 (time_of_day)
OUTPUT_COLUMNS = ["leak_locations", "leak_C"]

# ---------------------------------------------------------------------
# Operating-condition ranges (all "plausible", not extreme)
# ---------------------------------------------------------------------
DEMAND_FACTOR_RANGE        = (0.75, 1.30)   # global demand magnitude multiplier
DEMAND_SPATIAL_SIGMA       = 0.15           # per-junction log-normal spread (mean-preserving)
DEMAND_PATTERN_VARIANTS    = 3              # perturbed copies of every demand pattern in the network
RESERVOIR_HEAD_OFFSET_M    = 3.0            # +/- metres added to every reservoir head (common shift)
RESERVOIR_PATTERN_AMP      = 0.005          # +/- 0.5 % of head, smooth 24 h variation
TANK_LEVEL_FRACTION_RANGE  = (0.15, 0.90)   # initial level as fraction of (min_level .. max_level)
PIPE_ROUGHNESS_GLOBAL      = (0.85, 1.05)   # network-wide "ageing" factor on roughness
PIPE_ROUGHNESS_SIGMA       = 0.04           # per-pipe relative noise
MINOR_LOSS_GLOBAL          = (0.80, 1.30)
MINOR_LOSS_SIGMA           = 0.10
PUMP_SPEED_RANGE           = (0.90, 1.05)   # relative speed (VFD-like)
PUMP_OUTAGE_PROB           = 0.20           # probability a scenario contains a pump outage
VALVE_CLOSURE_PROB         = 0.10           # probability a scenario contains a valve closure

# Leak severity = equivalent orifice diameter + discharge coefficient
LEAK_DIAMETER_MM_RANGE     = (2.0, 15.0)    # sampled log-uniformly
LEAK_CD_RANGE              = (0.60, 0.80)
LEAK_MIN_PRESSURE_M        = 10.0           # leaks are only placed where pressure stays >= this all day
GRAVITY                    = 9.80665

# Quality gate: reject operating conditions that leave far more junctions at negative
# pressure than the untouched base network does (DDA would otherwise report "demand
# delivered" at unphysical negative pressures).
NEG_PRESSURE_MARGIN        = 0.03
REMOVE_LEAK_FLOW_FROM_DEMAND = False
# EPANET reports a node's "demand" INCLUDING its emitter (leak) flow. With False (default,
# literal use of the final EPANET results) the demand column at a leak junction is inflated
# by the leak flow. Set True to store consumer demand only (leak flow removed).

# ---------------------------------------------------------------------
# Objects taken from the notebook (names unchanged): path, base_wn, run_simulation,
# EMITTER_C_TO_SI, GPM_TO_M3S, np, pd, wntr, copy, os
# ---------------------------------------------------------------------
JUNCTIONS  = list(base_wn.junction_name_list)
PIPES      = list(base_wn.pipe_name_list)
RESERVOIRS = list(base_wn.reservoir_name_list)
TANKS      = list(base_wn.tank_name_list)
PUMPS      = list(base_wn.pump_name_list)
VALVES     = list(base_wn.valve_name_list)

actual_dims = dict(junctions=len(JUNCTIONS), pipes=len(PIPES), reservoirs=len(RESERVOIRS),
                   tanks=len(TANKS), pumps=len(PUMPS), valves=len(VALVES))
for key, expected_n in EXPECTED.items():
    assert actual_dims[key] == expected_n, (
        f"Network mismatch for {key}: loaded {actual_dims[key]}, expected {expected_n}")

N_J, N_P = len(JUNCTIONS), len(PIPES)
N_INPUT = 2 * N_J + N_P + 1
assert N_INPUT == EXPECTED_INPUT_DIM, f"Input dimension {N_INPUT} != {EXPECTED_INPUT_DIM}"

hyd = base_wn.options.hydraulic
EMITTER_EXPONENT = float(hyd.emitter_exponent)
assert EMITTER_EXPONENT == 0.5, (
    "Leak coefficient C = Cd*A*sqrt(2g) is only valid for an orifice exponent of 0.5")
IS_DDA = str(hyd.demand_model).upper() in ("DDA", "DD")
IS_HW = str(hyd.headloss).upper().startswith("H")
START_CLOCKTIME = float(base_wn.options.time.start_clocktime)
VALVE_TYPES = {v: base_wn.get_link(v).valve_type for v in VALVES}

# Unit sanity check using the notebook's own constants:
# an emitter of 1 GPM/psi^0.5 must give 1 GPM at 1 psi (= 0.70307 m of head).
_psi_in_m = 6894.757293168 / (1000.0 * GRAVITY)
assert abs(EMITTER_C_TO_SI * np.sqrt(_psi_in_m) - GPM_TO_M3S) / GPM_TO_M3S < 1e-3, \
    "EMITTER_C_TO_SI is inconsistent with GPM_TO_M3S"

# ---------------------------------------------------------------------
# Guard 1: the .inp file on disk must never change (read-only hash, no writes)
# ---------------------------------------------------------------------
def file_fingerprint(fp):
    st = os.stat(fp)
    with open(fp, "rb") as f:                      # read-only
        digest = hashlib.sha256(f.read()).hexdigest()
    return (st.st_size, st.st_mtime_ns, digest)

INP_FINGERPRINT_BEFORE = file_fingerprint(path)

# ---------------------------------------------------------------------
# Guard 2: base_wn must stay untouched (every scenario works on a deepcopy)
# ---------------------------------------------------------------------
def network_fingerprint(model):
    return (
        round(sum(model.get_link(n).roughness for n in PIPES), 6),
        round(sum(model.get_link(n).minor_loss or 0.0 for n in PIPES), 6),
        round(sum(ts.base_value for n in JUNCTIONS
                  for ts in model.get_node(n).demand_timeseries_list), 12),
        tuple(round(model.get_node(n).head_timeseries.base_value, 6) for n in RESERVOIRS),
        tuple(round(model.get_node(n).init_level, 6) for n in TANKS),
        tuple(round(model.get_link(n).speed_timeseries.base_value, 6) for n in PUMPS),
        tuple(str(model.get_link(n).initial_status) for n in PUMPS + VALVES),
        sum(1 for n in JUNCTIONS if (model.get_node(n).emitter_coefficient or 0) != 0),
        len(model.pattern_name_list),
        len(list(model.controls())),
    )

BASE_FINGERPRINT_BEFORE = network_fingerprint(base_wn)
assert BASE_FINGERPRINT_BEFORE[7] == 0, \
    "base_wn already contains emitters; leak labels would be ambiguous"

# ---------------------------------------------------------------------
# Graph helper: which junctions can still be reached from a reservoir/tank
# once closed pumps / valves / pipes are removed?
# ---------------------------------------------------------------------
def unsupplied_junctions(model):
    G = nx.Graph()
    G.add_nodes_from(model.node_name_list)
    for ln in model.link_name_list:
        link = model.get_link(ln)
        if link.initial_status == wntr.network.LinkStatus.Closed:
            continue
        G.add_edge(link.start_node_name, link.end_node_name)
    sources = set(RESERVOIRS) | set(TANKS)
    reachable = set()
    for comp in nx.connected_components(G):
        if sources & comp:
            reachable |= comp
    return {j for j in JUNCTIONS if j not in reachable}

BASE_UNSUPPLIED = unsupplied_junctions(base_wn)   # normally empty

# ---------------------------------------------------------------------
# Baseline run of the untouched network with the notebook's run_simulation()
# ---------------------------------------------------------------------
baseline_results = run_simulation(copy.deepcopy(base_wn), hours=SIM_HOURS)
REPORT_TIMES = [t for t in baseline_results.node["pressure"].index if t < SIM_HOURS * 3600]
assert len(REPORT_TIMES) == SIM_HOURS, f"expected {SIM_HOURS} hourly rows, got {len(REPORT_TIMES)}"
BASE_NEG_FRAC = float((baseline_results.node["pressure"].loc[REPORT_TIMES, JUNCTIONS]
                       .to_numpy() < 0).mean())

# Column names of the 3755 model inputs (order: demand, pressure, flowrate, time_of_day)
INPUT_COLUMNS = ([f"demand_{j}" for j in JUNCTIONS]
                 + [f"pressure_{j}" for j in JUNCTIONS]
                 + [f"flowrate_{p}" for p in PIPES]
                 + ["time_of_day"])
assert len(INPUT_COLUMNS) == N_INPUT == EXPECTED_INPUT_DIM

print("Network dimensions OK :", actual_dims)
print("Valve types           :", dict(Counter(VALVE_TYPES.values())))
print("Pump types            :", dict(Counter(base_wn.get_link(p_).pump_type for p_ in PUMPS)))
print(f"Demand model={hyd.demand_model} | headloss={hyd.headloss} | emitter exponent={EMITTER_EXPONENT}")
print(f"Baseline: {len(REPORT_TIMES)} hourly rows/scenario, negative-pressure fraction={BASE_NEG_FRAC:.4f}, "
      f"unsupplied junctions={len(BASE_UNSUPPLIED)}")
print("Input dimension       :", N_INPUT)

Network dimensions OK : {'junctions': 1242, 'pipes': 1270, 'reservoirs': 4, 'tanks': 15, 'pumps': 17, 'valves': 56}
Valve types           : {'PRV': 56}
Pump types            : {'POWER': 17}
Demand model=DDA | headloss=H-W | emitter exponent=0.5
Baseline: 24 hourly rows/scenario, negative-pressure fraction=0.0044, unsupplied junctions=0
Input dimension       : 3755


### 2. operating-condition variations + leak helpers

In [19]:
class ScenarioRejected(Exception):
    """Scenario is physically unusable (disconnected / heavy negative pressure / NaN)."""


def _unique_pattern_name(model, stem):
    name_ = stem
    k = 0
    while name_ in model.pattern_name_list:
        k += 1
        name_ = f"{stem}_{k}"
    return name_


def _remove_controls_targeting(model, link_names):
    """
    The network has tank-level controls such as
        IF TANK T-1 LEVEL BELOW x THEN PUMP ~@Pump-13 STATUS IS OPEN
    Such a control would silently re-open a pump/valve that we closed for an outage,
    so controls whose action targets a closed link are removed (in memory only).
    """
    targets = set(link_names)
    removed = []
    for cname, ctrl in list(model.controls()):
        try:
            hit = any(act.target()[0].name in targets for act in ctrl.actions())
        except Exception:
            hit = False
        if hit:
            model.remove_control(cname)
            removed.append(cname)
    return removed


def _perturb_pattern(base_m, rng):
    """Time shift + amplitude change + hourly noise, re-normalised to the original mean
    (so total daily volume is controlled only by the global demand factor)."""
    m = np.roll(np.asarray(base_m, dtype=float), int(rng.integers(-1, 2)))
    mean = m.mean()
    m = mean + rng.uniform(0.85, 1.20) * (m - mean)
    m = m * rng.normal(1.0, 0.04, size=m.shape)
    m = np.clip(m, 0.05 * mean, None)
    return m * (mean / m.mean())


def apply_demand_variation(model, rng, log):
    factor = float(rng.uniform(*DEMAND_FACTOR_RANGE))
    existing = set(model.pattern_name_list)
    default_pat = model.options.hydraulic.pattern
    default_pat = str(default_pat) if default_pat is not None else None

    def pat_of(ts):
        pn = ts.pattern_name if ts.pattern_name is not None else default_pat
        return pn if (pn is not None and pn in existing) else None

    used = sorted({pat_of(ts) for j in JUNCTIONS
                   for ts in model.get_node(j).demand_timeseries_list} - {None})
    variants = {}
    for pn in used:                                  # originals are never modified
        base_m = model.get_pattern(pn).multipliers
        names = []
        for k in range(DEMAND_PATTERN_VARIANTS):
            new = _unique_pattern_name(model, f"{pn}__dv{k}")
            model.add_pattern(new, _perturb_pattern(base_m, rng).tolist())
            names.append(new)
        variants[pn] = names

    s = DEMAND_SPATIAL_SIGMA
    for j in JUNCTIONS:
        for ts in model.get_node(j).demand_timeseries_list:
            ts.base_value = float(ts.base_value * factor * rng.lognormal(-0.5 * s * s, s))
            pn = pat_of(ts)
            if pn is not None:
                ts.pattern_name = variants[pn][int(rng.integers(len(variants[pn])))]
    log["demand_factor"] = factor


def apply_reservoir_variation(model, rng, log):
    common = float(rng.uniform(-RESERVOIR_HEAD_OFFSET_M, RESERVOIR_HEAD_OFFSET_M))
    offsets = {}
    for r in RESERVOIRS:                             # several reservoirs are handled independently
        res = model.get_node(r)
        off = float(np.clip(common + rng.normal(0.0, 0.5),
                            -RESERVOIR_HEAD_OFFSET_M - 1.0, RESERVOIR_HEAD_OFFSET_M + 1.0))
        res.head_timeseries.base_value = float(res.head_timeseries.base_value + off)
        offsets[r] = round(off, 3)

        # head pattern: reservoir head = base_value * multiplier(t)
        old = res.head_timeseries.pattern_name
        if old is not None and old in model.pattern_name_list:
            base_m = np.asarray(model.get_pattern(old).multipliers, dtype=float)
        else:
            base_m = np.ones(SIM_HOURS)
        amp = rng.uniform(0.0, RESERVOIR_PATTERN_AMP)
        phase = rng.uniform(0.0, 2 * np.pi)
        hrs = np.arange(len(base_m))
        shape = (1.0 + amp * np.sin(2 * np.pi * hrs / len(base_m) + phase)
                 + rng.normal(0.0, amp / 4.0 + 1e-12, size=len(base_m)))
        new = _unique_pattern_name(model, f"RESHEAD_{r}")
        model.add_pattern(new, (base_m * shape).tolist())
        res.head_timeseries.pattern_name = new
    log["reservoir_offsets_m"] = offsets


def apply_tank_variation(model, rng, log):
    for t in TANKS:
        tank = model.get_node(t)
        lo, hi = tank.min_level, tank.max_level
        if hi - lo > 1e-6:
            tank.init_level = float(lo + rng.uniform(*TANK_LEVEL_FRACTION_RANGE) * (hi - lo))


def apply_pipe_variation(model, rng, log):
    age = float(rng.uniform(*PIPE_ROUGHNESS_GLOBAL))
    k_glob = float(rng.uniform(*MINOR_LOSS_GLOBAL))
    for pn in PIPES:
        pipe = model.get_link(pn)
        r0 = pipe.roughness
        r = r0 * age * rng.normal(1.0, PIPE_ROUGHNESS_SIGMA)
        if IS_HW:   # Hazen-Williams C is dimensionless; keep it in a realistic band
            r = float(np.clip(r, min(40.0, r0), max(150.0, r0)))
        else:
            r = float(max(r, 1e-9))
        pipe.roughness = r
        if pipe.minor_loss and pipe.minor_loss > 0:  # only where a minor loss already exists
            pipe.minor_loss = float(max(0.0, pipe.minor_loss * k_glob
                                        * rng.normal(1.0, MINOR_LOSS_SIGMA)))
    log["roughness_factor"] = round(age, 4)
    log["minor_loss_factor"] = round(k_glob, 4)


def apply_pump_variation(model, rng, log):
    # EPANET applies the speed setting to power pumps as well (flow scales ~ linearly with speed).
    for pn in PUMPS:
        sp = model.get_link(pn).speed_timeseries
        sp.base_value = float(np.clip(sp.base_value * rng.uniform(*PUMP_SPEED_RANGE), 0.5, 1.2))

    outaged = []
    if rng.random() < PUMP_OUTAGE_PROB:
        want = 1 if (rng.random() < 0.8 or len(PUMPS) < 2) else 2
        for idx in rng.permutation(len(PUMPS)):
            if len(outaged) == want:
                break
            pn = PUMPS[int(idx)]
            pump = model.get_link(pn)
            prev = pump.initial_status
            pump.initial_status = wntr.network.LinkStatus.Closed
            if unsupplied_junctions(model) <= BASE_UNSUPPLIED:   # outage must not strand junctions
                outaged.append(pn)
            else:
                pump.initial_status = prev
        _remove_controls_targeting(model, outaged)
    log["pump_outages"] = outaged


def apply_valve_variation(model, rng, log):
    """
    Valve 'setting' means something different for every valve type, so each type gets its
    own rule (settings are in SI units inside WNTR):
      PRV / PSV : downstream / upstream pressure setpoint [m]  -> +/-15 %
      PBV       : constant pressure drop [m]                   -> +/-15 %
      FCV       : flow limit [m3/s]                            -> +/-20 %
      TCV       : minor-loss coefficient K [-]                 -> 0.7x .. 1.5x
      GPV       : setting is a head-loss CURVE id, not a number -> never touched (closure only)
    """
    for vn in VALVES:
        valve = model.get_link(vn)
        vt, s0 = VALVE_TYPES[vn], valve.initial_setting
        if vt in ("PRV", "PSV", "PBV"):
            valve.initial_setting = float(s0 * rng.uniform(0.85, 1.15))
        elif vt == "FCV":
            valve.initial_setting = float(s0 * rng.uniform(0.80, 1.20))
        elif vt == "TCV":
            valve.initial_setting = float(s0 * rng.uniform(0.70, 1.50))

    closed = []
    if rng.random() < VALVE_CLOSURE_PROB and len(VALVES) > 0:
        for idx in rng.permutation(len(VALVES))[:5]:
            vn = VALVES[int(idx)]
            valve = model.get_link(vn)
            prev = valve.initial_status
            valve.initial_status = wntr.network.LinkStatus.Closed
            if unsupplied_junctions(model) <= BASE_UNSUPPLIED:
                closed.append(vn)
                break
            valve.initial_status = prev
        _remove_controls_targeting(model, closed)
    log["valve_closures"] = closed


# ---------------------------------------------------------------------
# Leaks.  EPANET emitter:  Q_leak = C * P^0.5   (P = pressure head [m], Q [m3/s])
# Orifice equation:        Q = Cd * A * sqrt(2 g h)   ->   C = Cd * A * sqrt(2 g)
# so the leak "severity" is the equivalent hole diameter and the stored C is in the
# same SI units that WNTR's junction.emitter_coefficient uses.
# (C / EMITTER_C_TO_SI would give the same value in EPANET's GPM/psi^0.5 INP units.)
# ---------------------------------------------------------------------
def leak_coefficient(diameter_m, cd):
    area = np.pi * diameter_m ** 2 / 4.0
    return float(f"{cd * area * np.sqrt(2.0 * GRAVITY):.6e}")   # 7 significant digits


def leak_candidates(noleak_results):
    P = noleak_results.node["pressure"].loc[REPORT_TIMES, JUNCTIONS]
    min_p = P.min(axis=0)
    ok = [j for j in JUNCTIONS if j not in BASE_UNSUPPLIED and min_p[j] >= LEAK_MIN_PRESSURE_M]
    return ok


def select_leaks(rng, n_leaks, candidates):
    if len(candidates) < n_leaks:
        raise ScenarioRejected(f"only {len(candidates)} well-pressurised junctions for {n_leaks} leaks")
    picks = rng.choice(len(candidates), size=n_leaks, replace=False)
    locations = sorted(candidates[int(i)] for i in picks)
    lo, hi = np.log(LEAK_DIAMETER_MM_RANGE[0]), np.log(LEAK_DIAMETER_MM_RANGE[1])
    leak_C = {}
    for j in locations:
        d_mm = float(np.exp(rng.uniform(lo, hi)))
        leak_C[j] = leak_coefficient(d_mm / 1000.0, float(rng.uniform(*LEAK_CD_RANGE)))
    return locations, leak_C


def apply_leaks(model, leak_locations, leak_C):
    for j in leak_locations:
        model.get_node(j).emitter_coefficient = leak_C[j]
    # verification: each leak junction carries exactly its C, nothing else has an emitter
    for j in leak_locations:
        assert model.get_node(j).emitter_coefficient == leak_C[j], f"emitter mismatch at {j}"
    n_emitters = sum(1 for j in JUNCTIONS if (model.get_node(j).emitter_coefficient or 0) != 0)
    assert n_emitters == len(leak_locations), "unexpected emitters present in the model"

### 3. gates, verification, extraction, one-scenario runner

In [20]:
def quality_gate(results, neg_limit):
    P = results.node["pressure"].loc[REPORT_TIMES, JUNCTIONS].to_numpy(dtype=float)
    D = results.node["demand"].loc[REPORT_TIMES, JUNCTIONS].to_numpy(dtype=float)
    Q = results.link["flowrate"].loc[REPORT_TIMES, PIPES].to_numpy(dtype=float)
    if not (np.isfinite(P).all() and np.isfinite(D).all() and np.isfinite(Q).all()):
        raise ScenarioRejected("NaN/inf in EPANET results")
    neg = float((P < 0).mean())
    if neg > neg_limit:
        raise ScenarioRejected(f"negative-pressure fraction {neg:.3f} > limit {neg_limit:.3f}")
    return neg


def verify_closed_links(results, log):
    for ln in log["pump_outages"] + log["valve_closures"]:
        q = np.abs(results.link["flowrate"].loc[REPORT_TIMES, ln].to_numpy(dtype=float))
        assert q.max() < 1e-6, f"closed link {ln} still carries flow ({q.max():.2e} m3/s)"


def verify_leak_hydraulics(final, noleak, leak_locations, leak_C):
    """
    Under demand-driven analysis (DDA) consumer demand is fixed, so
        reported_demand(final) - reported_demand(no leak) == emitter flow == C * P^0.5
    at the leak junctions and == 0 everywhere else. This proves the C stored in leak_C is
    the C that EPANET actually used.
    Note: in some networks EPANET's *t = 0* report row can show an emitter flow that does not
    match the reported pressure (initial-solution quirk); later steps match exactly. So a few
    isolated steps may deviate, but the C must match in >= 75 % of the hourly steps.
    Returns the number of deviating (junction, timestep) pairs for the scenario log.
    """
    if not IS_DDA:
        return 0
    d_f = final.node["demand"].loc[REPORT_TIMES, JUNCTIONS]
    d_0 = noleak.node["demand"].loc[REPORT_TIMES, JUNCTIONS]
    p_f = final.node["pressure"].loc[REPORT_TIMES, JUNCTIONS]
    excess = d_f - d_0
    others = [j for j in JUNCTIONS if j not in set(leak_locations)]
    assert np.allclose(excess[others].to_numpy(dtype=float), 0.0, atol=1e-8), \
        "demand changed at junctions without a leak"
    n_off = 0
    for j in leak_locations:
        expected = leak_C[j] * np.clip(p_f[j].to_numpy(dtype=float), 0.0, None) ** EMITTER_EXPONENT
        match = np.isclose(excess[j].to_numpy(dtype=float), expected, rtol=5e-3, atol=1e-7)
        assert match.mean() >= 0.75, f"emitter flow at {j} does not match C*sqrt(P)"
        n_off += int((~match).sum())
    return n_off


def extract_block(final, noleak, leak_locations, rng):
    """Return a (24, 3755) array: [demand(1242) | pressure(1242) | flowrate(1270) | time_of_day]."""
    # .loc with explicit lists guarantees the column order and raises KeyError if anything is missing
    demand = np.array(final.node["demand"].loc[REPORT_TIMES, JUNCTIONS], dtype=float)   # writable copy
    pressure = final.node["pressure"].loc[REPORT_TIMES, JUNCTIONS].to_numpy(dtype=float)
    flow = final.link["flowrate"].loc[REPORT_TIMES, PIPES].to_numpy(dtype=float)
    if REMOVE_LEAK_FLOW_FROM_DEMAND and leak_locations:
        idx = [JUNCTIONS.index(j) for j in leak_locations]
        demand[:, idx] = noleak.node["demand"].loc[REPORT_TIMES, leak_locations].to_numpy(dtype=float)
    tod = ((np.asarray(REPORT_TIMES, dtype=float) + START_CLOCKTIME) % 86400.0) / 3600.0
    X = np.hstack([demand, pressure, flow, tod[:, None]])
    assert X.shape == (len(REPORT_TIMES), N_INPUT), f"bad block shape {X.shape}"

    # spot-check 30 random cells against the raw EPANET result tables
    for _ in range(30):
        r = int(rng.integers(len(REPORT_TIMES)))
        t = REPORT_TIMES[r]
        jx, px = int(rng.integers(N_J)), int(rng.integers(N_P))
        if not REMOVE_LEAK_FLOW_FROM_DEMAND:
            assert X[r, jx] == final.node["demand"].at[t, JUNCTIONS[jx]]
        assert X[r, N_J + jx] == final.node["pressure"].at[t, JUNCTIONS[jx]]
        assert X[r, 2 * N_J + px] == final.link["flowrate"].at[t, PIPES[px]]
    return X


def run_one_scenario(sc_idx, attempt, n_leaks):
    rng = np.random.default_rng([SEED, sc_idx, attempt])   # reproducible per (scenario, attempt)
    log = {}
    n_off = 0

    wn = copy.deepcopy(base_wn)                             # FIXED topology, fresh copy every time

    # ---- operating conditions (all in memory; the .inp file is never touched) ----
    apply_demand_variation(wn, rng, log)
    apply_reservoir_variation(wn, rng, log)
    apply_tank_variation(wn, rng, log)
    apply_pipe_variation(wn, rng, log)
    apply_pump_variation(wn, rng, log)
    apply_valve_variation(wn, rng, log)
    if not (unsupplied_junctions(wn) <= BASE_UNSUPPLIED):
        raise ScenarioRejected("junctions disconnected from every source")

    # ---- run 1: operating conditions only ----
    noleak = run_simulation(wn, hours=SIM_HOURS)
    neg0 = quality_gate(noleak, BASE_NEG_FRAC + NEG_PRESSURE_MARGIN)

    # ---- run 2: add leak(s) and re-run; final results are the dataset observations ----
    if n_leaks == 0:
        final, leak_locations, leak_C = noleak, [], {}
    else:
        leak_locations, leak_C = select_leaks(rng, n_leaks, leak_candidates(noleak))
        wn_leak = copy.deepcopy(wn)
        apply_leaks(wn_leak, leak_locations, leak_C)
        final = run_simulation(wn_leak, hours=SIM_HOURS)
        quality_gate(final, neg0 + NEG_PRESSURE_MARGIN)
        n_off = verify_leak_hydraulics(final, noleak, leak_locations, leak_C)

    assert set(leak_locations) == set(leak_C.keys()), "leak_locations / leak_C key mismatch"
    assert len(leak_locations) == n_leaks
    verify_closed_links(final, log)
    X = extract_block(final, noleak, leak_locations, rng)

    log.update(n_leaks=n_leaks, mean_pressure_m=round(float(X[:, N_J:2 * N_J].mean()), 3),
               neg_pressure_frac=round(neg0, 4), leak_check_deviating_steps=n_off,
               leak_flow_gpm_at_30m=round(sum(c * np.sqrt(30.0) for c in leak_C.values()) / GPM_TO_M3S, 2))
    return X, leak_locations, leak_C, log

### 4. generate the dataset, build the dataframe, final checks

In [21]:
# Failures that are normal for random operating conditions: scenario gate, EPANET
# non-convergence (RuntimeError from convergence_error=True) or an EPANET error code.
EXPECTED_FAILURES = (ScenarioRejected, RuntimeError, wntr.epanet.exceptions.EpanetException)

master_rng = np.random.default_rng(SEED)
# balanced leak counts (0,1,2,3 repeated) in random order
leak_count_plan = master_rng.permutation(np.resize([0, 1, 2, 3], N_SCENARIOS))

blocks, row_leak_locations, row_leak_C, scenario_id_per_row = [], [], [], []
scenario_log_rows, failed_attempts = [], []
n_success = 0
t_start = time.time()

for sc in range(N_SCENARIOS):
    n_leaks = int(leak_count_plan[sc])
    for attempt in range(MAX_ATTEMPTS_PER_SCENARIO):
        try:
            X, locs, leak_C, log = run_one_scenario(sc, attempt, n_leaks)
        except EXPECTED_FAILURES as exc:             # EPANET failure / rejected scenario -> resample
            failed_attempts.append((sc, attempt, f"{type(exc).__name__}: {str(exc)[:120]}"))
            continue
        # Anything else (AssertionError from a verification, or a coding error) is NOT swallowed.
        blocks.append(X)
        row_leak_locations.extend([json.dumps(locs)] * X.shape[0])
        row_leak_C.extend([json.dumps(leak_C)] * X.shape[0])
        scenario_id_per_row.extend([sc] * X.shape[0])
        scenario_log_rows.append(dict(scenario_id=sc, attempt=attempt, **log,
                                      leak_locations=locs, leak_C=leak_C))
        n_success += 1
        break
    else:
        print(f"[warn] scenario {sc} failed {MAX_ATTEMPTS_PER_SCENARIO} times and was skipped")
    if (sc + 1) % 10 == 0:
        print(f"  {sc + 1}/{N_SCENARIOS} scenarios processed "
              f"({n_success} ok, {len(failed_attempts)} rejected attempts, {time.time() - t_start:.0f}s)")

assert n_success > 0, "no scenario succeeded"

# ---- assemble the dataframe: 3755 input columns + 2 output columns ----
X_df = pd.DataFrame(np.vstack(blocks), columns=INPUT_COLUMNS)
Y_df = pd.DataFrame({"leak_locations": row_leak_locations, "leak_C": row_leak_C})
dataset_df = pd.concat([X_df, Y_df], axis=1)

# Kept OUTSIDE dataset_df so the dataframe has exactly 3755 + 2 columns.
# Use scenario_ids for GroupKFold / train-test splits (the 24 rows of a scenario are correlated).
scenario_ids = pd.Series(scenario_id_per_row, name="scenario_id", index=dataset_df.index)
scenario_log = pd.DataFrame(scenario_log_rows)

# ---- final verification ----
assert dataset_df.shape[1] == EXPECTED_INPUT_DIM + 2
assert list(dataset_df.columns[:EXPECTED_INPUT_DIM]) == INPUT_COLUMNS
assert list(dataset_df.columns[EXPECTED_INPUT_DIM:]) == OUTPUT_COLUMNS
assert dataset_df.shape[0] == n_success * len(REPORT_TIMES)
assert not dataset_df[INPUT_COLUMNS].isna().any().any(), "NaN in inputs"
assert np.isfinite(dataset_df[INPUT_COLUMNS].to_numpy()).all(), "inf in inputs"
assert dataset_df["time_of_day"].between(0, 24, inclusive="left").all()

for loc_json, c_json in dataset_df[OUTPUT_COLUMNS].drop_duplicates().itertuples(index=False):
    locs_ = json.loads(loc_json)
    c_ = json.loads(c_json)
    assert isinstance(locs_, list) and isinstance(c_, dict)
    assert set(locs_) == set(c_.keys()), "set(leak_locations) != set(leak_C.keys())"
    assert all(j in set(JUNCTIONS) for j in locs_), "leak placed on a non-junction"
    assert len(locs_) <= 3

assert file_fingerprint(path) == INP_FINGERPRINT_BEFORE, ".inp file was modified!"
assert network_fingerprint(base_wn) == BASE_FINGERPRINT_BEFORE, "base_wn was modified!"

print("\n" + "=" * 70)
print("DATASET GENERATION COMPLETE")
print("=" * 70)
print("Dataset shape        :", dataset_df.shape)
print("Input dimension      :", len(INPUT_COLUMNS), f"({N_J} + {N_J} + {N_P} + 1)")
print("Output column names  :", [c for c in dataset_df.columns if c in OUTPUT_COLUMNS])
print(f"Successful scenarios : {n_success} / {N_SCENARIOS}  "
      f"(rejected/failed attempts that were resampled: {len(failed_attempts)})")
print("Leak-count mix       :", dict(Counter(scenario_log["n_leaks"])))
print(".inp file unchanged  : True | base_wn unchanged: True")

  10/500 scenarios processed (10 ok, 11 rejected attempts, 22s)
  20/500 scenarios processed (20 ok, 13 rejected attempts, 38s)
  30/500 scenarios processed (30 ok, 27 rejected attempts, 65s)
  40/500 scenarios processed (40 ok, 34 rejected attempts, 101s)
  50/500 scenarios processed (50 ok, 42 rejected attempts, 123s)
[warn] scenario 50 failed 12 times and was skipped
  60/500 scenarios processed (59 ok, 60 rejected attempts, 161s)
  70/500 scenarios processed (69 ok, 69 rejected attempts, 185s)
  80/500 scenarios processed (79 ok, 81 rejected attempts, 216s)
  90/500 scenarios processed (89 ok, 93 rejected attempts, 248s)
  100/500 scenarios processed (99 ok, 104 rejected attempts, 278s)
  110/500 scenarios processed (109 ok, 113 rejected attempts, 297s)
  120/500 scenarios processed (119 ok, 117 rejected attempts, 323s)
  130/500 scenarios processed (129 ok, 138 rejected attempts, 381s)
  140/500 scenarios processed (139 ok, 140 rejected attempts, 395s)
  150/500 scenarios processe

## Save dataset to a file

In [22]:
train_dataset = dataset_df.sample(frac=0.8, random_state=42)
test_dataset = dataset_df.drop(train_dataset.index)

train_dataset.to_csv(
    r"C:\Data\pipeline-leakage-system\Phase 1\pipeline_leak_dataset(train).csv",
    index=False
)

test_dataset.to_csv(
    r"C:\Data\pipeline-leakage-system\Phase 1\pipeline_leak_dataset(test).csv",
    index=False
)

print(train_dataset.shape)
print(test_dataset.shape)

(9562, 3757)
(2390, 3757)


## Removing the $2^{\text{nd}}$ last redundant column

In [ ]:
df = pd.read_csv(r"C:\Data\pipeline-leakage-system\Phase 1\pipeline_leak_dataset(train).csv")

df = df.drop(df.columns[-2], axis=1)

df.to_csv(r"C:\Data\pipeline-leakage-system\Phase 1\output(train).csv", index=False)

In [ ]:
df = pd.read_csv(r"C:\Data\pipeline-leakage-system\Phase 1\pipeline_leak_dataset(test).csv")

df = df.drop(df.columns[-2], axis=1)

df.to_csv(r"C:\Data\pipeline-leakage-system\Phase 1\output(test).csv", index=False)

## Expand the output into a vector (Multiple Columns)

In [3]:
import json
import os

folder = r"C:\Data\pipeline-leakage-system\Phase 1"

def expand_leak_C(in_name, out_name):
    df = pd.read_csv(os.path.join(folder, in_name))
    assert df.columns[-1] == "leak_C"

    # junction order taken from the demand_ columns (same 1242 junctions, same order)
    junctions = [c[len("demand_"):] for c in df.columns if c.startswith("demand_")]
    assert len(junctions) == 1242

    leak_dicts = df["leak_C"].apply(json.loads)
    unknown = set().union(*[d.keys() for d in leak_dicts]) - set(junctions)
    assert not unknown, f"leak on unknown junctions: {unknown}"

    # one column per junction: its leak coefficient C, or 0.0 if it has no leak
    leak_cols = pd.DataFrame(
        [[d.get(j, 0.0) for j in junctions] for d in leak_dicts],
        columns=[f"leak_C_{j}" for j in junctions],
        index=df.index,
    )
    out = pd.concat([df.drop(columns="leak_C"), leak_cols], axis=1)
    assert out.shape[1] == 3755 + 1242

    out.to_csv(os.path.join(folder, out_name), index=False)
    print(out_name, out.shape)

expand_leak_C("pipeline_leak_dataset(train).csv", "pipeline_leak_dataset_expanded(train).csv")
expand_leak_C("pipeline_leak_dataset(test).csv",  "pipeline_leak_dataset_expanded(test).csv")

pipeline_leak_dataset_expanded(train).csv (9562, 4997)
pipeline_leak_dataset_expanded(test).csv (2390, 4997)
